# Capstone 3 — Hospital Operations & Readmission Analytics
### Microsoft Fabric Medallion Capstone

**Business scenario:** "Riverside Health Network" wants to track bed utilization, length-of-stay, and flag patients at high risk of a 30-day readmission.

**Fabric capability highlighted:** Lakehouse Medallion architecture with a **referential-integrity-aware Silver layer** and a Gold layer built specifically to support a readmission-risk Power BI report.

**What this notebook builds:**
1. Synthetic patients, admissions, and diagnoses data
2. Bronze → Silver (with quarantine for orphan/invalid admissions) → Gold
3. `gold.mart_readmission_risk` — a rules-based readmission risk flag per patient
4. `gold.mart_ward_utilization` — daily bed occupancy by ward

Attach this notebook to a Lakehouse (e.g. `hospital_capstone_lakehouse`) before running.

> Note: all data below is synthetic and generated for training/demo purposes only — no real patient data is used.

In [ ]:
%pip install faker --quiet

In [ ]:
import random
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql import functions as F
from faker import Faker

fake = Faker()
Faker.seed(21)
random.seed(21)

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

NUM_PATIENTS = 4000
NUM_ADMISSIONS = 7000
WARDS = ["Cardiology", "Orthopedics", "General Medicine", "ICU", "Maternity", "Oncology"]
DIAGNOSES = ["Heart Failure", "Pneumonia", "Hip Fracture", "Diabetes Complication",
             "COPD Exacerbation", "Sepsis", "Post-Surgical Recovery"]

## 1. Generate source data

In [ ]:
patients = [{
    "patient_id": f"PT{i:06d}",
    "age": random.randint(0, 95),
    "gender": random.choice(["M", "F"]),
    "city": fake.city(),
    "has_chronic_condition": random.random() < 0.3,
} for i in range(1, NUM_PATIENTS + 1)]
df_patients = pd.DataFrame(patients)

valid_patient_ids = df_patients["patient_id"].tolist()
start_date = datetime.utcnow() - timedelta(days=365)

admissions = []
for i in range(1, NUM_ADMISSIONS + 1):
    admit_date = start_date + timedelta(days=random.randint(0, 360))
    los_days = max(1, int(random.gauss(4, 3)))
    admissions.append({
        "admission_id": f"ADM{i:07d}",
        "patient_id": random.choice(valid_patient_ids),
        "ward": random.choice(WARDS),
        "diagnosis": random.choice(DIAGNOSES),
        "admit_date": admit_date.date().isoformat(),
        "discharge_date": (admit_date + timedelta(days=los_days)).date().isoformat(),
        "length_of_stay_days": los_days,
    })

df_admissions = pd.DataFrame(admissions)

# Inject ~1% orphan admissions (patient_id not in the patient master) -> tests Silver RI check
orphan_idx = df_admissions.sample(frac=0.01, random_state=2).index
df_admissions.loc[orphan_idx, "patient_id"] = "PT999999"

print(f"{len(df_patients):,} patients | {len(df_admissions):,} admissions "
      f"({len(orphan_idx)} intentional orphans)")

## 2. Bronze layer

In [ ]:
def to_bronze(pdf, table_name):
    sdf = spark.createDataFrame(pdf).withColumn("_ingestion_timestamp", F.current_timestamp())
    sdf.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
    print(f"{table_name}: {sdf.count():,} rows")

to_bronze(df_patients, "bronze.patients")
to_bronze(df_admissions, "bronze.admissions")

## 3. Silver layer — referential integrity + type casting

In [ ]:
silver_patients = spark.table("bronze.patients").dropDuplicates(["patient_id"])
silver_patients.write.format("delta").mode("overwrite").saveAsTable("silver.patients")

bronze_adm = (spark.table("bronze.admissions")
    .withColumn("admit_date", F.to_date("admit_date"))
    .withColumn("discharge_date", F.to_date("discharge_date"))
    .withColumn("length_of_stay_days", F.col("length_of_stay_days").cast("int"))
    .dropDuplicates(["admission_id"]))

valid_patients = silver_patients.select("patient_id")
passed_adm = bronze_adm.join(valid_patients, "patient_id", "left_semi")
failed_adm = (bronze_adm.join(valid_patients, "patient_id", "left_anti")
    .withColumn("_dq_reason", F.lit("patient_id not found in silver.patients")))

failed_adm.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable("silver.admissions_quarantine")
passed_adm.write.format("delta").mode("overwrite").saveAsTable("silver.admissions")

print(f"silver.admissions: {passed_adm.count():,} passed | {failed_adm.count():,} quarantined")

## 4. Gold layer — star schema

In [ ]:
dim_patient = silver_patients
dim_patient.write.format("delta").mode("overwrite").saveAsTable("gold.dim_patient")

# 30-day readmission flag: same patient, another admission within 30 days of a prior discharge
w_adm = spark.table("silver.admissions").select("patient_id", "admission_id", "admit_date", "discharge_date")

fact_admissions = (w_adm.alias("a")
    .join(w_adm.alias("b"), (F.col("a.patient_id") == F.col("b.patient_id")) &
                             (F.col("b.admit_date") > F.col("a.discharge_date")) &
                             (F.datediff(F.col("b.admit_date"), F.col("a.discharge_date")) <= 30),
          "left")
    .groupBy("a.admission_id")
    .agg(F.max(F.col("b.admission_id").isNotNull()).alias("is_readmission_within_30d"))
    .join(spark.table("silver.admissions"), "admission_id"))

fact_admissions.write.format("delta").mode("overwrite").saveAsTable("gold.fact_admissions")
print(f"gold.fact_admissions: {fact_admissions.count():,} rows "
      f"({fact_admissions.filter('is_readmission_within_30d').count():,} flagged as 30-day readmissions)")

## 5. Business marts: readmission risk & ward utilization

In [ ]:
mart_readmission_risk = (fact_admissions
    .join(dim_patient, "patient_id")
    .groupBy("patient_id", "age", "has_chronic_condition")
    .agg(F.count("*").alias("total_admissions_1y"),
         F.sum(F.col("is_readmission_within_30d").cast("int")).alias("readmissions_within_30d"),
         F.avg("length_of_stay_days").alias("avg_length_of_stay"))
    .withColumn("readmission_risk_flag",
                (F.col("readmissions_within_30d") >= 1) |
                ((F.col("has_chronic_condition")) & (F.col("total_admissions_1y") >= 3))))

mart_readmission_risk.write.format("delta").mode("overwrite").saveAsTable("gold.mart_readmission_risk")

mart_ward_utilization = (fact_admissions
    .groupBy("ward", "admit_date")
    .agg(F.count("*").alias("admissions_that_day"),
         F.avg("length_of_stay_days").alias("avg_length_of_stay")))

mart_ward_utilization.write.format("delta").mode("overwrite").saveAsTable("gold.mart_ward_utilization")

print(f"gold.mart_readmission_risk: {mart_readmission_risk.count():,} patients "
      f"({mart_readmission_risk.filter('readmission_risk_flag').count():,} flagged high-risk)")
print(f"gold.mart_ward_utilization: {mart_ward_utilization.count():,} rows")

## 6. Capstone checkpoint
Suggested Power BI pages: **Readmission Risk** (patient list filtered to `readmission_risk_flag = true`), **Ward Utilization** (daily admissions trend by ward), and a **Length of Stay** distribution by diagnosis.

In [ ]:
for t in ["gold.dim_patient", "gold.fact_admissions", "gold.mart_readmission_risk", "gold.mart_ward_utilization"]:
    print(f"{t:30s} -> {spark.table(t).count():,} rows")
print("\nCapstone 3 (Hospital Operations & Readmission Analytics) complete.")